In [0]:
# encode loan statuses to 0 and 1

from pyspark.sql import functions as F

bronze = spark.table("mlops_project.lendingclub_bronze")
good = ["Fully Paid"]
bad  = ["Charged Off", "Default", "Late (31-120 days)", "Late (16-30 days)"]

df = (bronze
      .withColumn("label_default",
                  F.when(F.col("loan_status").isin(bad), F.lit(1))
                   .when(F.col("loan_status").isin(good), F.lit(0))
                   .otherwise(F.lit(None)).cast("int"))
      .filter(F.col("label_default").isNotNull())
)

df.show(5)



+----+---------+---------+-----------+---------------+----------+--------+-----------+-----+---------+--------------------+----------+--------------+----------+-------------------+--------+-----------+----------+----+----+------------------+--------------------+--------+----------+-----+-----------+----------------+--------------+----------------------+----------------------+--------+-------+---------+----------+---------+-------------------+---------+-------------+---------------+---------------+---------------+-------------+------------------+----------+-----------------------+------------+---------------+------------+------------------+--------------------------+---------------------------+-----------+----------------+----------------+---------+-------------------------+--------------+------------+-----------+-----------+-----------+-----------+-----------+------------------+------------+-------+-----------+-----------+----------+--------+----------------+------+-----------+--------

In [0]:
# drop columns with information not available at the time of actual application
leakage_cols = [
    "loan_status",
    "out_prncp","out_prncp_inv",
    "total_pymnt","total_pymnt_inv",
    "total_rec_prncp","total_rec_int","total_rec_late_fee",
    "recoveries","collection_recovery_fee",
    "last_pymnt_d","last_pymnt_amnt","next_pymnt_d",
    "last_credit_pull_d",
    "tot_coll_amt","tot_cur_bal","open_acc_6m","open_act_il","open_il_12m","open_il_24m",
    "mths_since_last_delinq","mths_since_last_record"  # optional; keep if you want
]
existing_leakage = [c for c in leakage_cols if c in df.columns]
df = df.drop(*existing_leakage)

# drop all-null columns
non_null_counts = df.select([F.count(F.col(c)).alias(c) for c in df.columns]).collect()[0].asDict()
all_null_cols = [c for c, cnt in non_null_counts.items() if cnt == 0]
df = df.drop(*all_null_cols)

drop_cols = [c for c in df.columns if c.startswith("hardship_")] + ["pymnt_plan"] + [
    "emp_title","desc","title","zip_code",
    "earliest_cr_line","sec_app_earliest_cr_line",
    "issue_d","settlement_date","debt_settlement_flag_date",
    "hardship_end_date","hardship_start_date",
    "payment_plan_start_date","hardship_reason",
    "hardship_loan_status"
]

df = df.drop(*drop_cols)

df = df.drop("_source_file")  # up to you

In [0]:
(df.write
   .format("delta")
   .mode("overwrite")
   .saveAsTable("mlops_project.lendingclub_silver")
)